In [1]:
import pandas as pd

In [5]:
al = pd.read_excel("AL_Data.xlsx")
lo = pd.read_excel("LO_Data.xlsx")

In [9]:
print("American Lumber shape:", al.shape)
print("Lane One shape:", lo.shape)

American Lumber shape: (15559, 46)
Lane One shape: (8167, 19)


# Data Cleaning

<h3>Standardize Column Names</h3> 

In [13]:
al.columns = al.columns.str.strip().str.replace(" ", "_").str.replace("#", "Num")
lo.columns = lo.columns.str.strip().str.replace(" ", "_").str.replace("#", "Num")

In [29]:
print("AL columns:", al.columns)
print("LaneOne columns:", lo.columns)

AL columns: Index(['poRecordID', 'AljexProNum', 'VendorCID', 'Mill', 'BuyerName',
       'ContractPurchase', 'poType', 'LinkToNo', 'Salesman', 'FreightPO',
       'ShipVia', 'ItemID', 'ItemDescription', 'ConvBF', 'Grade', 'Species',
       'PurchasePrice', 'LumberCost', 'FreightFactor', 'PurchaseUOM', 'BF',
       'CallReadyDate', 'PickUpDate', 'DeliveredDate', 'Active', 'DATRate',
       'EstFreight', 'ActFreightCost', 'CarrierCost', 'FreightOutByLine',
       'ReceivingFee', 'ContributionFee', 'TotalCost', 'SalePrice', 'SaleUOM',
       'OriginAddress1', 'OriginAddress2', 'OriginCity', 'OriginST',
       'OriginZip', 'CustomerName', 'DestAddress1', 'DestAddress2', 'DestCity',
       'DestST', 'DestZip'],
      dtype='object')
LaneOne columns: Index(['Pro_Num', 'Type_of_Shipment', 'Actual_Dispatcher', 'Created_Date',
       'Ship_Date', 'Delivered_Date', 'Pickup_City', 'Pickup_State',
       'Pickup_Zip', 'Consignee_City', 'Consignee_State', 'Consignee_Zip',
       'Carrier_Name', 'Mi

In [37]:
# American Lumber date columns
date_cols_al = ["CallReadyDate", "PickUpDate", "DeliveredDate"]
for col in date_cols_al:
    al[col] = pd.to_datetime(al[col], errors="coerce")

# LaneOne date columns
date_cols_lane = ["Created_Date", "Ship_Date", "Delivered_Date", "Mill_Ready_Dt"]
for col in date_cols_lane:
    lo[col] = pd.to_datetime(lo[col], errors="coerce")

In [41]:
# AL numeric fields
num_cols_al = ["PurchasePrice", "LumberCost", "FreightFactor", "BF", 
               "EstFreight", "ActFreightCost", "CarrierCost", 
               "FreightOutByLine", "ReceivingFee", "ContributionFee", "TotalCost", "SalePrice"]

for col in num_cols_al:
    if col in al.columns:
        al[col] = pd.to_numeric(al[col], errors="coerce")

# LaneOne numeric fields
num_cols_lane = ["Miles/Class", "LOT_Revenue", "LOT_Carrier_Expense", 
                 "LOT_Gross_Profit", "LOT_Profit_%"]

for col in num_cols_lane:
    if col in lo.columns:
        lo[col] = pd.to_numeric(lo[col], errors="coerce")


<h3> Missing Values </h3>

In [46]:
print("Missing values in AL:\n", al.isna().sum().sort_values(ascending=False))
print("\nMissing values in LaneOne:\n", lo.isna().sum().sort_values(ascending=False))

Missing values in AL:
 FreightFactor       15559
DestAddress2        15402
OriginAddress2      15328
ReceivingFee        11100
FreightPO            6312
PickUpDate           5611
CarrierCost          5523
AljexProNum          5519
SaleUOM              4462
SalePrice            4462
CustomerName         4462
DATRate              4462
ContributionFee      4462
LinkToNo             4460
Salesman             4354
Grade                 903
DestZip               270
DestAddress1          267
DestCity              267
DestST                267
CallReadyDate          42
TotalCost               3
EstFreight              3
ActFreightCost          2
FreightOutByLine        2
LumberCost              2
OriginCity              1
OriginST                1
OriginZip               1
OriginAddress1          1
poRecordID              1
Active                  1
BF                      1
PurchaseUOM             1
PurchasePrice           1
Species                 1
ConvBF                  1
ItemDescription

In [48]:
al_no_pro = al[al["AljexProNum"].isna()]

print("Rows with AljexProNum = NULL:", al_no_pro.shape[0])
print("\nSample records:\n", al_no_pro.head(10))


Rows with AljexProNum = NULL: 5519

Sample records:
      poRecordID  AljexProNum  VendorCID  \
30     480805.0          NaN   372983.0   
69     478353.0          NaN    14310.0   
118    483159.0          NaN   373929.0   
133    483212.0          NaN      972.0   
134    483212.0          NaN      972.0   
136    480806.0          NaN   372983.0   
137    480807.0          NaN   372983.0   
165    483282.0          NaN     1021.0   
205    484662.0          NaN      972.0   
206    484687.0          NaN      975.0   

                                               Mill     BuyerName  \
30                          K&R Sawmill (Viola, AR)   Mays, Grady   
69       Weyerhaeuser NR Company (Philadelphia, MS)  Oney, Donnie   
118              Interfor US, Inc (Bay Springs, MS)  Oney, Donnie   
133            Georgia-Pacific Corp. (Pineland, TX)  Oney, Donnie   
134            Georgia-Pacific Corp. (Pineland, TX)  Oney, Donnie   
136                         K&R Sawmill (Viola, AR)   Mays,

In [15]:
print("Unique AL loads:", al["AljexProNum"].nunique())
print("Total AL rows:", al.shape[0])

Unique AL loads: 6351
Total AL rows: 15559


In [17]:
print("Unique LaneOne loads:", lo["Pro_Num"].nunique())
print("Total LaneOne rows:", lo.shape[0])

Unique LaneOne loads: 8167
Total LaneOne rows: 8167


In [23]:
set_al = set(al["AljexProNum"].unique())
set_lane = set(lo["Pro_Num"].unique())

In [25]:
in_both = len(set_al & set_lane)
only_in_al = len(set_al - set_lane)
only_in_lane = len(set_lane - set_al)

print("Total unique AL loads:", len(set_al))
print("Total unique LaneOne loads:", len(set_lane))
print("Loads in BOTH datasets:", in_both)
print("Loads ONLY in AL:", only_in_al)
print("Loads ONLY in LaneOne:", only_in_lane)

# Optional: percentage overlap
print("Percent of AL loads found in LaneOne:", round(in_both / len(set_al) * 100, 2), "%")
print("Percent of LaneOne loads found in AL:", round(in_both / len(set_lane) * 100, 2), "%")

Total unique AL loads: 6352
Total unique LaneOne loads: 8167
Loads in BOTH datasets: 5758
Loads ONLY in AL: 594
Loads ONLY in LaneOne: 2409
Percent of AL loads found in LaneOne: 90.65 %
Percent of LaneOne loads found in AL: 70.5 %


In [63]:
columns_to_keep = [
    'AljexProNum', 'poRecordID', 'Mill', 'VendorCID',
    'ContractPurchase', 'poType', 'ShipVia', 'LinkToNo',
    'ItemID', 'ItemDescription', 'BF', 'PurchasePrice',
    'LumberCost', 'FreightOutByLine', 'ContributionFee',
    'TotalCost', 'SalePrice', 'CallReadyDate',
    'PickUpDate', 'DeliveredDate'
]

In [67]:
al_subset = al[columns_to_keep]

# Aggregate per load
al_agg = al_subset.groupby('AljexProNum').agg({
    # Numeric sums
    'LumberCost': 'sum',
    'FreightOutByLine': 'sum',
    'ContributionFee': 'sum',
    'TotalCost': 'sum',
    'SalePrice': 'sum',
    'BF': 'sum',
    
    # Numeric averages
    'PurchasePrice': 'mean',
    
    # Categorical / ID: take first or combine unique
    'poRecordID': lambda x: ', '.join(x.dropna().astype(str)),
    'Mill': 'first',
    'VendorCID': 'first',
    'ContractPurchase': 'first',
    'poType': 'first',
    'ShipVia': 'first',
    'LinkToNo': lambda x: ', '.join(x.dropna().astype(str)),
    'ItemID': lambda x: ', '.join(x.dropna().astype(str)),
    'ItemDescription': lambda x: ', '.join(x.dropna().astype(str)),
    
    # Dates
    'CallReadyDate': 'min',
    'PickUpDate': 'min',
    'DeliveredDate': 'max'
}).reset_index()


print(al_agg.head())

   AljexProNum  LumberCost  FreightOutByLine  ContributionFee  TotalCost  \
0      32350.0  4033.12000            900.00           104.96   5038.080   
1      33639.0  3245.18213           1670.50            63.36   4979.042   
2      33750.0  5588.83500            878.58           122.03   6589.445   
3      33769.0  4126.13000           1055.31           112.64   5294.080   
4      33786.0  4053.37524           1290.24           101.92   5445.535   

   SalePrice       BF  PurchasePrice          poRecordID  \
0     594.00  23039.0        235.000  481460.0, 481460.0   
1       5.45  12672.0        387.916            483228.0   
2     320.00  23295.0        265.000            483431.0   
3     310.00  22528.0        230.000            484107.0   
4     295.00  21504.0        265.091            482471.0   

                                             Mill  VendorCID  \
0          LaSalle Lumber Company, LLC (Olla, LA)    15975.0   
1        D B Hostetler Sawmill (Warm Springs, AR)   37

In [69]:
merged = pd.merge(al_agg, lo, left_on="AljexProNum", right_on="Pro_Num", how="inner", suffixes=("_al", "_lane"))

In [71]:
print(merged)

      AljexProNum  LumberCost  FreightOutByLine  ContributionFee   TotalCost  \
0         33769.0  4126.13000        1055.31000           112.64  5294.08000   
1         33797.0  4240.81500        1195.00000           110.94  5546.75500   
2         33830.0  4324.99900        1000.00085             0.00  5700.00000   
3         33832.0  4324.99900        1000.00085             0.00  5700.00000   
4         33835.0  3668.20714         673.00000           117.33  4458.53719   
...           ...         ...               ...              ...         ...   
5753      43005.0  4372.00000        1100.00000           121.60  5593.60000   
5754      43006.0  6554.72000         900.00000           116.48  7571.20000   
5755      43009.0  4463.05000         900.00000            43.12  5406.17000   
5756      43015.0  3593.69000         799.00000           122.03  4514.71973   
5757      43024.0  2057.80176        3066.00000           116.48  5240.28174   

      SalePrice       BF  PurchasePrice

In [77]:
merged.to_csv('merged_al_lo_data.csv', index=False)

In [79]:
merged.to_excel('merged_al_lo_data.xlsx', index=False)

In [61]:
print("AL total unique loads:", al["AljexProNum"].nunique())
print("LaneOne total unique loads:", lo["Pro_Num"].nunique())
print("Joined loads:", merged["AljexProNum"].nunique())
print("Joined rows:", merged.shape[0])

AL total unique loads: 6351
LaneOne total unique loads: 8167
Joined loads: 5758
Joined rows: 9113


In [73]:
print("AL aggregated rows:", len(al_agg))
print("LaneOne rows:", len(lo))
print("Merged rows:", len(merged))

AL aggregated rows: 6351
LaneOne rows: 8167
Merged rows: 5758


In [75]:
print("Unique AL loads in merged data:", merged['AljexProNum'].nunique())
print("Unique AL loads originally:", al_agg['AljexProNum'].nunique())

Unique AL loads in merged data: 5758
Unique AL loads originally: 6351
